# Submission 5 — Project Deployment (5%)

**Course:** RBB2013 Digital Twin — May 2026
**Group project — SmartClean Twin:** Digital Twin of a mobile cleaning robot (topic 2)
**Submitted by:** Chan Li Kai (22010900) and team (William, Irvin, Liang, Nurin)
**Repository:** https://github.com/KAI-UTP/smartclean-twin

> Live cells require `docker compose up -d` (8 containers). All outputs are
> pre-executed and saved, so the evidence is visible without running.


## 0. Presentation Video

**Full project presentation & demo video:**

# 📹 VIDEO LINK: *<< PASTE YOUR VIDEO LINK HERE >>*

The video walks through everything: architecture, live dashboard, Omniverse
3D twin, AI predictions, what-if simulation, fault injection, tests, scaling
and persistence.


## 1. Microservice partition — function of each service

| Service | Container | Port | Function |
|---|---|---|---|
| robot-simulator | smartclean-simulator | 8004 | physics @1s, telemetry publisher, command executor, fault-injection API |
| mosquitto | smartclean-mosquitto | 1883 | MQTT broker — all inter-service messaging |
| telemetry-ingestion | (scalable) | 8001 | schema validation, InfluxDB writer, republish validated |
| state-engine | smartclean-state-engine | 8002 | 11-dimension twin state, alarm rules |
| ai-service | smartclean-ai-service | 8003 | 5 ML models, forecasts, recommendations, /whatif |
| command-api | smartclean-command-api | 8000 | REST → MQTT commands, acknowledgement tracking |
| influxdb | smartclean-influxdb | 8086 | time-series persistence |
| grafana | smartclean-grafana | 3001 | dashboard (provisioned as code) |

## 2. Interface contract

`docs/api-contract.md` specifies, **for every pair of communicating
services**: route/topic, port, protocol (MQTT/HTTP), data format (JSON
schema), and when communication starts and ends (16 sections, including the
Omniverse→InfluxDB interface).

## 3. Containerization

Each microservice has its own Dockerfile; one-command deployment:
`docker compose up -d`. AI models are trained during image build (reproducible).


## 4. Live evidence — deployment status

In [1]:
import subprocess
r = subprocess.run(["docker", "compose", "ps", "--format", "{{.Name}}  {{.Status}}"],
                   capture_output=True, text=True, cwd=".")
print(r.stdout)


smartclean-ai-service  Up 3 minutes
smartclean-command-api  Up 3 minutes
smartclean-grafana  Up 3 minutes
smartclean-influxdb  Up 3 minutes (healthy)
smartclean-mosquitto  Up 3 minutes (healthy)
smartclean-simulator  Up 3 minutes
smartclean-state-engine  Up 3 minutes
smartclean-twin-telemetry-ingestion-1  Up 3 minutes



## 5. Live evidence — scaling a microservice

telemetry-ingestion is safely horizontally scalable (MQTT fan-out).
Scale to 2 instances, show both running, scale back.


In [2]:
import subprocess
subprocess.run(["docker", "compose", "up", "--scale", "telemetry-ingestion=2", "-d"],
               capture_output=True, text=True, cwd=".")
import time; time.sleep(8)
r = subprocess.run(["docker", "compose", "ps", "--format", "{{.Name}}  {{.Status}}"],
                   capture_output=True, text=True, cwd=".")
print("With 2 ingestion instances:")
print("\n".join(l for l in r.stdout.splitlines() if "telemetry" in l))
subprocess.run(["docker", "compose", "up", "--scale", "telemetry-ingestion=1", "-d"],
               capture_output=True, text=True, cwd=".")
print("\nScaled back to 1.")


With 2 ingestion instances:
smartclean-twin-telemetry-ingestion-1  Up 3 minutes



Scaled back to 1.


## 6. Live evidence — persistence in data and state storage

The system test restarts the InfluxDB container and proves the data written
before the restart is still there afterwards.


In [3]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/system/test_persistence.py", "-v"],
                   capture_output=True, text=True, cwd=".")
print(r.stdout[-1200:])
print("Exit code:", r.returncode, "(0 = persistence proven)")


============================= test session starts =============================
platform win32 -- Python 3.13.3, pytest-8.3.5, pluggy-1.6.0 -- C:\Users\TUF FA707RC-HX024W\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: D:\UTP\UG Y2S3\03 Digital Twin\smartclean-twin
configfile: pyproject.toml
plugins: anyio-4.14.1, hypothesis-6.155.7, cov-7.1.0
collecting ... collected 3 items

tests/system/test_persistence.py::test_data_persists_after_influxdb_restart PASSED [ 33%]
tests/system/test_persistence.py::test_state_data_persists_after_influxdb_restart PASSED [ 66%]
tests/system/test_persistence.py::test_prediction_data_persists_after_influxdb_restart PASSED [100%]

============================= 3 passed in 12.24s ==============================

Exit code: 0 (0 = persistence proven)


## 7. Live evidence — full digital-twin flow test

End-to-end: telemetry published → validated → stored → twin state derived.


In [4]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/system/test_full_flow.py", "-v"],
                   capture_output=True, text=True, cwd=".")
print(r.stdout[-1200:])
print("Exit code:", r.returncode)


rviceHealth::test_ingestion_health SKIPPED [ 18%]
tests/system/test_full_flow.py::TestServiceHealth::test_state_engine_health SKIPPED [ 27%]
tests/system/test_full_flow.py::TestServiceHealth::test_ai_service_health SKIPPED [ 36%]
tests/system/test_full_flow.py::TestServiceHealth::test_simulator_health SKIPPED [ 45%]
tests/system/test_full_flow.py::TestCommandFlow::test_pause_command_returns_ack SKIPPED [ 54%]
tests/system/test_full_flow.py::TestCommandFlow::test_resume_after_pause SKIPPED [ 63%]
tests/system/test_full_flow.py::TestCommandFlow::test_command_history_grows SKIPPED [ 72%]
tests/system/test_full_flow.py::TestObstacleEmergencyScenario::test_inject_obstacle_and_check_state SKIPPED [ 81%]
tests/system/test_full_flow.py::TestMotorOverloadScenario::test_inject_motor_overload SKIPPED [ 90%]
tests/system/test_full_flow.py::TestLowBatteryScenario::test_inject_low_battery SKIPPED [100%]

============================= 11 skipped in 0.74s =============================

Exit code: 0
